# Bitcoin competition: file inspection
Run the cells from top to bottom. This notebook reads the four original CSV files.

We inspect file sizes, columns, inferred data types, missing values, and the exact label strings stored in the CSV. Required packages: `pandas`, `numpy`, and a Python Jupyter kernel.

## Findings Summary for this Notebook

1. 110,537 of 141,772 training rows (77.97%) have no label
2. Among these rows with labels (31,235 rows): 27,591 licit and 3,644 illicit (only 11.7%) -> for ordinary supervised training
3. Dataset is imbalanced since illicit rows (only 11.7% of all labelled rows)
4. `test.csv` has 15,329 transactions -> predictions are needed (can refer to the sample)
5. `txs_edgelist.csv`has 234,355 connections

In [1]:
from pathlib import Path
import csv
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display

# Find the project when running from its root or a subfolder.
DATA_FOLDER = "bt-4012-competition-2026"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / DATA_FOLDER).is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"Open this notebook inside the project containing {DATA_FOLDER}.")
DATA = ROOT / DATA_FOLDER
filenames = ["train.csv", "test.csv", "txs_edgelist.csv", "sample_submission.csv"]
missing_files = [name for name in filenames if not (DATA / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing files: {missing_files}")
print("Data folder:", DATA.name)

Data folder: bt-4012-competition-2026


## 1. Load the four files
`read_csv` normally infers numeric types and converts blank cells and recognized missing-value tokens to `NaN`. Later, we read labels separately with that conversion disabled to distinguish the actual stored strings.

In [2]:
tables = {name: pd.read_csv(DATA / name, low_memory=False) for name in filenames}
train = tables["train.csv"]
test = tables["test.csv"]
edges = tables["txs_edgelist.csv"]
sample_submission = tables["sample_submission.csv"]

overview = pd.DataFrame([
    {"file": name, "rows": len(df), "columns": len(df.columns),
     "size_MB": round((DATA / name).stat().st_size / 1_000_000, 2),
     "missing_cells": int(df.isna().sum().sum())}
    for name, df in tables.items()
])
display(overview)
for name, df in tables.items():
    print(f"\n{name}: first 3 rows (wide tables show a selection of columns)")
    display(df.head(3))

,file,rows,columns,size_MB,missing_cells
0,train.csv,141772,168,460.98,110537
1,test.csv,15329,168,49.58,0
2,txs_edgelist.csv,234355,2,4.47,0
3,sample_submission.csv,15329,2,0.14,0



train.csv: first 3 rows (wide tables show a selection of columns)


,txId,time_step,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,...,feat_157,feat_158,feat_159,feat_160,feat_161,feat_162,feat_163,feat_164,feat_165,label
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.12197,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,NaN
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.12197,-0.043875,-0.113002,-0.061584,-0.162112,...,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,NaN
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.12197,-0.043875,-0.113002,-0.061584,-0.162749,...,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792,NaN



test.csv: first 3 rows (wide tables show a selection of columns)


,index,txId,time_step,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_156,feat_157,feat_158,feat_159,feat_160,feat_161,feat_162,feat_163,feat_164,feat_165
0,0,200459281,36,-0.172879,-0.058452,0.463609,-0.121970,-0.043875,-0.113002,-0.061584,...,2.262621,1.947206,1.461330,1.461369,-0.098889,-0.08749,-0.084674,-0.140597,-1.760926,-1.760984
1,1,199944769,36,-0.172828,0.339970,1.018602,0.178180,-0.043875,0.222447,-0.061584,...,-0.524789,-0.531617,-0.979074,-0.978556,0.018279,-0.08749,-0.131155,-0.097524,-0.120613,-0.119792
2,2,198827545,36,-0.172282,0.837998,1.018602,0.553368,-0.043875,0.641758,-0.061584,...,1.149152,1.070472,1.461330,1.461369,-0.098889,-0.08749,-0.084674,-0.140597,-1.760926,-1.760984



txs_edgelist.csv: first 3 rows (wide tables show a selection of columns)


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870



sample_submission.csv: first 3 rows (wide tables show a selection of columns)


,index,target
0,0,0.5
1,1,0.5
2,2,0.5


## Findings from Section 1
The files use `feat_1`–`feat_165` (rather than `feature_1`–`feature_165`). Test data also has an `index` column

The submission uses `index,target`. Keep identifiers `index` from the test.csv

We inspect raw headers too, because pandas can rename duplicate or blank column headers when loading.

## 2. Data types and missing values
The per-column report includes every column. `missing_count` counts values pandas recognizes as missing; whitespace-only strings are checked separately. Infinity is reported separately because it is not counted by `isna()`.

A numeric-looking identifier is still an identifier. Missing labels are different from missing predictors and must not be filled with the licit class.

In [4]:
column_reports = {}
for name, df in tables.items():
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null_count": df.notna().sum(),
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(3),
    })
    report.index.name = "column"
    report["whitespace_only_count"] = 0
    for col in df.select_dtypes(include=["object", "string"]).columns:
        report.loc[col, "whitespace_only_count"] = int(
            df[col].astype("string").str.strip().eq("").fillna(False).sum())
    report["infinite_count"] = 0
    for col in df.select_dtypes(include="number").columns:
        report.loc[col, "infinite_count"] = int(np.isinf(df[col]).sum())
    column_reports[name] = report
    print(f"\n{name}: inferred type counts")
    display(df.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("columns"))
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(report)
    problems = report.loc[(report["missing_count"] > 0) |
                          (report["whitespace_only_count"] > 0) |
                          (report["infinite_count"] > 0)]
    print("Columns with missing, blank, or infinite values:", problems.index.tolist())

for name in ["train.csv", "test.csv"]:
    df = tables[name]
    non_numeric = [c for c in feature_columns if c in df and
                   not pd.api.types.is_numeric_dtype(df[c])]
    print(f"{name}: non-numeric feature columns = {non_numeric}")


train.csv: inferred type counts


,columns
dtype,
float64,166
int64,2


,dtype,non_null_count,missing_count,missing_percent,whitespace_only_count,infinite_count
column,,,,,,
txId,int64,141772,0,0.000,0,0
time_step,int64,141772,0,0.000,0,0
feat_1,float64,141772,0,0.000,0,0
feat_2,float64,141772,0,0.000,0,0
feat_3,float64,141772,0,0.000,0,0
feat_4,float64,141772,0,0.000,0,0
feat_5,float64,141772,0,0.000,0,0
feat_6,float64,141772,0,0.000,0,0
feat_7,float64,141772,0,0.000,0,0


Columns with missing, blank, or infinite values: ['label']

test.csv: inferred type counts


,columns
dtype,
float64,165
int64,3


...

,dtype,non_null_count,missing_count,missing_percent,whitespace_only_count,infinite_count
column,,,,,,
txId1,int64,234355,0,0.0,0,0
txId2,int64,234355,0,0.0,0,0


Columns with missing, blank, or infinite values: []

sample_submission.csv: inferred type counts


,columns
dtype,
int64,1
float64,1


,dtype,non_null_count,missing_count,missing_percent,whitespace_only_count,infinite_count
column,,,,,,
index,int64,15329,0,0.0,0,0
target,float64,15329,0,0.0,0,0


Columns with missing, blank, or infinite values: []
train.csv: non-numeric feature columns = []
test.csv: non-numeric feature columns = []


## Findings from Section 2

All the 110537 missing_cells from `train.csv` is from the column `label`, indicating there are 110537 unlabelled rows

## 3. How are unknown labels actually stored?
Read only `label` again as text with missing-value conversion disabled. `repr` puts quotes around each stored value, so `''` means an empty CSV field and `'unknown'` would mean literal text.

The normalized categories below are for inspection only. Known labels are numeric 0 (licit) and 1 (illicit). Blank/none/unknown/null/NaN/NA/N/A tokens are reported as unknown; any other value is flagged for investigation. The original `train` table is unchanged.

In [5]:
raw_labels = pd.read_csv(
    DATA / "train.csv", usecols=["label"], dtype="string", na_filter=False
)["label"]
label_tokens = raw_labels.value_counts(dropna=False).rename_axis("raw_label").reset_index(name="count")
label_tokens["stored_value_repr"] = label_tokens["raw_label"].map(repr)
label_tokens["percent_of_train"] = (100 * label_tokens["count"] / len(raw_labels)).round(3)
display(label_tokens[["stored_value_repr", "count", "percent_of_train"]])

print("Label dtype after normal read_csv:", train["label"].dtype)
display(train["label"].value_counts(dropna=False).rename_axis("parsed_label").to_frame("count"))

normalized = raw_labels.str.strip().str.lower()
numeric_labels = pd.to_numeric(normalized, errors="coerce")
unknown_mask = normalized.isin(["", "none", "unknown", "null", "nan", "na", "n/a"])
label_status = pd.Series("UNEXPECTED: investigate", index=train.index, name="label_status")
label_status.loc[unknown_mask] = "unknown"
label_status.loc[numeric_labels.eq(0)] = "licit (0)"
label_status.loc[numeric_labels.eq(1)] = "illicit (1)"
display(label_status.value_counts().to_frame("transactions"))

unexpected_mask = label_status.eq("UNEXPECTED: investigate")
if unexpected_mask.any():
    print("Unexpected stored labels:", raw_labels[unexpected_mask].unique().tolist())
print("Examples of unknown-label rows:")
display(train.loc[unknown_mask, ["txId", "time_step", "label"]].head(5))
print("Unknown labels are not licit labels. Exclude unknowns from supervised loss and scoring.")

,stored_value_repr,count,percent_of_train
0,'',110537,77.968
1,'0',27591,19.462
2,'1',3644,2.57


Label dtype after normal read_csv: float64


,count
parsed_label,
NaN,110537
0.0,27591
1.0,3644


,transactions
label_status,
unknown,110537
licit (0),27591
illicit (1),3644


Examples of unknown-label rows:


column,txId,time_step,label
0,230425980,1,NaN
1,5530458,1,NaN
2,232022460,1,NaN
4,230460314,1,NaN
5,230459870,1,NaN


Unknown labels are not licit labels. Exclude unknowns from supervised loss and scoring.


## 4. Inspection summary
Use these outputs to decide the next preparation steps. No rows have been removed, missing values filled, or labels overwritten.

In [6]:
print(f"Training transactions: {len(train):,}")
print(f"Test transactions: {len(test):,}")
print(f"Edges: {len(edges):,}")
print(f"Unknown labels: {unknown_mask.sum():,}")
print(f"Unexpected labels to investigate: {unexpected_mask.sum():,}")
for name in ["train.csv", "test.csv"]:
    available = [c for c in feature_columns if c in tables[name]]
    missing = int(tables[name][available].isna().sum().sum())
    print(f"{name}: {len(available)} expected features present; {missing:,} missing feature values")
print("Next: inspect label counts by time step before choosing a validation cutoff.")

Training transactions: 141,772
Test transactions: 15,329
Edges: 234,355
Unknown labels: 110,537
Unexpected labels to investigate: 0
train.csv: 165 expected features present; 0 missing feature values
test.csv: 165 expected features present; 0 missing feature values
Next: inspect label counts by time step before choosing a validation cutoff.
